# Credit Risk Default Prediction

**Objective:** predict the probability that a consumer-finance applicant will default
within 12 months (`Default 12 Flag`), using application, demographic, employment and
credit-bureau information available at the time of application.

**Model:** LightGBM (gradient-boosted trees), trained on engineered ratio/flag
features plus smoothed, PSI-checked target encodings of the categorical fields.

**Evaluation metric:** ROC-AUC, measured on a time-based hold-out so the score
reflects how the model would perform on applications that arrive *after* the
training window — closer to a real production setting than a random split.

**Validation AUC:** 0.6485

This number, the SHAP driver ranking, and the decile lift are also reproduced
live by the **Executive Summary** cell at the end of this notebook — re-running
the pipeline regenerates them from scratch, so this header and that summary
should always agree.

A separate notebook, `Credit_Risk_Ensemble_Experiment.ipynb`, covers the CatBoost +
XGBoost stacking ensemble explored during experimentation. It's kept out of this
notebook on purpose: a single, well-validated model is easier to explain end to end.


## 1. Imports and config

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

import lightgbm as lgb
import shap

warnings.filterwarnings("ignore")

# ---- single source of truth for paths / target / seed --------------------
DATA_DIR = "."                       # change this one line if your files live elsewhere
TARGET = "Default 12 Flag"
ID_COL = "ID"
current_seed = 42

np.random.seed(current_seed)


## 2. Data loading

In [ ]:
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")
META_PATH  = os.path.join(DATA_DIR, "metaData.xlsx")

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

if os.path.exists(META_PATH):
    try:
        meta_sheets = pd.read_excel(META_PATH, sheet_name=None)
        print("metaData.xlsx sheets:", list(meta_sheets.keys()))
    except Exception as e:
        print("Could not read metaData.xlsx:", e)
else:
    print("metaData.xlsx not found at", META_PATH, "(optional, skipping).")


## 3. Metadata analysis

In [ ]:
def column_summary(df):
    """One row per column: dtype, missing %, cardinality."""
    rows = []
    for c in df.columns:
        ser = df[c]
        rows.append({
            "column": c,
            "dtype": str(ser.dtype),
            "missing": int(ser.isna().sum()),
            "missing_pct": round(float(ser.isna().mean()), 4),
            "unique": int(ser.nunique(dropna=True)),
        })
    return pd.DataFrame(rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)

train.info()

print("\nTarget balance (train):")
print(train[TARGET].value_counts(normalize=True).rename("share").to_frame())

print("\nTop 15 columns by missing %:")
column_summary(train).head(15)


## 4. Feature engineering

One reusable function builds every engineered feature: date parsing, age, ratios,
declared-vs-actual mismatch flags, zero/missing flags, capped + log-transformed
ratios, a company-size proxy, channel flags, and a PSI-safe grouping of the
high-cardinality `JIS Address Code` column (top categories kept, the long tail
collapsed into `OTHER`).

The grouping is **fit on train only** (`jis_keep_set`) and re-used on test, so there
is no leakage and the resulting category is stable enough to target-encode safely.


In [ ]:
def safe_to_numeric(series):
    """Strip thousands separators / stray characters from money-like text columns."""
    s = series.astype(str).str.replace(",", "").str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "NULL": np.nan})
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")


def parse_dates(df):
    df = df.copy()
    if "Application Date" in df.columns:
        df["Application Date"] = pd.to_datetime(df["Application Date"], errors="coerce")
    if "Application Time" in df.columns and "Application Date" in df.columns:
        df["Application Time"] = df["Application Time"].astype(str).str.zfill(6)
        df["application_datetime"] = pd.to_datetime(
            df["Application Date"].dt.strftime("%Y-%m-%d") + " " +
            df["Application Time"].str.slice(0, 2) + ":" +
            df["Application Time"].str.slice(2, 4) + ":" +
            df["Application Time"].str.slice(4, 6),
            errors="coerce",
        )
    if "Date of Birth" in df.columns and "Application Date" in df.columns:
        df["Date of Birth"] = pd.to_datetime(df["Date of Birth"], errors="coerce")
        age_years = (df["Application Date"] - df["Date of Birth"]).dt.days / 365.25
        df["age"] = age_years.replace([np.inf, -np.inf], np.nan).round().astype("Int64")
    return df


def engineer_features(df, target_col=None, jis_keep_set=None):
    """
    Single reusable feature-engineering step.

    Call with jis_keep_set=None on the training set (the grouping is learned and
    returned); pass the returned set back in when calling on test, so test never
    influences the grouping.
    """
    df = df.copy()

    # money-like columns -> numeric
    money_cols = ["Rent Burden Amount", "Total Annual Income",
                  "Declared Amount of Unsecured Loans", "Amount of Unsecured Loans",
                  "Application Limit Amount(Desired)"]
    for c in money_cols:
        if c in df.columns:
            df[c] = safe_to_numeric(df[c])

    # ratios
    if {"Rent Burden Amount", "Total Annual Income"} <= set(df.columns):
        df["rent_burden_ratio"] = df["Rent Burden Amount"] / (df["Total Annual Income"] / 12 + 1e-9)
    if {"Amount of Unsecured Loans", "Total Annual Income"} <= set(df.columns):
        df["unsecured_to_income"] = df["Amount of Unsecured Loans"] / (df["Total Annual Income"] + 1e-9)
    if {"Application Limit Amount(Desired)", "Total Annual Income"} <= set(df.columns):
        df["limit_to_income"] = df["Application Limit Amount(Desired)"] / (df["Total Annual Income"] + 1e-9)

    # declared-vs-actual mismatch flags
    if {"Declared Number of Unsecured Loans", "Number of Unsecured Loans"} <= set(df.columns):
        df["declared_num_mismatch"] = (
            df["Declared Number of Unsecured Loans"] != df["Number of Unsecured Loans"]
        ).astype(int)
    if {"Declared Amount of Unsecured Loans", "Amount of Unsecured Loans"} <= set(df.columns):
        df["declared_amt_mismatch"] = (
            df["Declared Amount of Unsecured Loans"].fillna(0) != df["Amount of Unsecured Loans"].fillna(0)
        ).astype(int)
        df["declared_amt_diff"] = (
            df["Declared Amount of Unsecured Loans"].fillna(0) - df["Amount of Unsecured Loans"].fillna(0)
        )

    # zero / missing flags
    if "Total Annual Income" in df.columns:
        df["income_missing"] = df["Total Annual Income"].isna().astype(int)
        df["income_is_zero"] = (df["Total Annual Income"].fillna(0) == 0).astype(int)
    if "Rent Burden Amount" in df.columns:
        df["rent_is_zero"] = (df["Rent Burden Amount"].fillna(0) == 0).astype(int)
    if "Amount of Unsecured Loans" in df.columns:
        df["unsecured_is_zero"] = (df["Amount of Unsecured Loans"].fillna(0) == 0).astype(int)

    # company size numeric proxy
    if "Company Size Category" in df.columns:
        company_map = {1: 200, 2: 1000, 3: 500, 4: 100, 5: 50, 6: 20, 7: 5, 8: 3, 9: 1}
        df["company_size_num"] = df["Company Size Category"].map(company_map).fillna(0)

    # channel / device flags
    if "Reception Type Category" in df.columns:
        df["is_mobile_app"] = df["Reception Type Category"].isin([1701, 1801]).astype(int)
    if "Major Media Code" in df.columns:
        df["is_internet_channel"] = (df["Major Media Code"] == 11).astype(int)

    # clean infinities, flag negatives, cap at p99, log1p-transform
    for c in ["rent_burden_ratio", "unsecured_to_income", "limit_to_income"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df.loc[~np.isfinite(df[c]), c] = np.nan
            df[c + "_neg"] = (df[c] < 0).astype(int)
            p99 = df[c].quantile(0.99)
            cap_val = float(p99) if np.isfinite(p99) else np.nan
            df[c + "_cap"] = df[c].clip(upper=cap_val)
            df["log1p_" + c] = np.log1p(df[c + "_cap"].clip(lower=0))

    # application hour / suspicious time-of-day
    if "application_datetime" in df.columns:
        df["app_hour"] = df["application_datetime"].dt.hour.fillna(-1).astype(int)
        df["suspicious_app_time"] = df["app_hour"].isin([0, 1, 2, 3]).astype(int)

    # PSI-safe grouping for the high-cardinality JIS Address Code column
    if "JIS Address Code" in df.columns:
        if jis_keep_set is None:
            freq_keep = set(df["JIS Address Code"].value_counts().head(40).index)
            mean_keep = set()
            if target_col is not None and target_col in df.columns:
                agg = df.groupby("JIS Address Code")[target_col].agg(["mean", "count"])
                mean_keep = set(
                    agg[agg["count"] >= 10].sort_values("mean", ascending=False).head(10).index
                )
            jis_keep_set = freq_keep | mean_keep
        df["JIS_grouped"] = df["JIS Address Code"].apply(
            lambda x: x if x in jis_keep_set else "OTHER"
        ).astype(object)

    return df, jis_keep_set


In [ ]:
train = parse_dates(train)
test  = parse_dates(test)

train_fe, jis_keep_set = engineer_features(train, target_col=TARGET, jis_keep_set=None)
test_fe,  _            = engineer_features(test,  target_col=None,   jis_keep_set=jis_keep_set)

print("Engineered train shape:", train_fe.shape)
print("Engineered test shape :", test_fe.shape)

preview_cols = [c for c in [
    ID_COL, "age", "company_size_num", "is_internet_channel", "is_mobile_app",
    "declared_num_mismatch", "declared_amt_mismatch", "declared_amt_diff",
    "log1p_unsecured_to_income", "log1p_rent_burden_ratio", "log1p_limit_to_income",
    "suspicious_app_time", "JIS_grouped",
] if c in train_fe.columns]
train_fe[preview_cols].head()


## 5. Target encoding

One clean sequence: fit a smoothed, fold-wise (out-of-fold) target encoding on
train, transform test with the full-train mapping, check the Population
Stability Index (PSI) between train and test for every encoded column, then drop
any column unstable enough to risk leaking train-only signal into the model.


In [ ]:
def fold_target_encode(df, col, target_col, n_splits=5, seed=42, smoothing=30):
    """Out-of-fold, mean-smoothed target encoding (no leakage on the train rows)."""
    df = df.copy()
    df[col] = df[col].astype(object)
    oof = pd.Series(index=df.index, dtype=float)
    prior = df[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        tr, val = df.iloc[tr_idx], df.iloc[val_idx]
        agg = tr.groupby(col)[target_col].agg(["mean", "count"])
        smooth = (agg["count"] * agg["mean"] + smoothing * prior) / (agg["count"] + smoothing)
        oof.iloc[val_idx] = val[col].map(smooth).fillna(prior)
    full_agg = df.groupby(col)[target_col].agg(["mean", "count"])
    full_smooth = (full_agg["count"] * full_agg["mean"] + smoothing * prior) / (full_agg["count"] + smoothing)
    return oof, full_smooth.to_dict(), prior


def psi(expected, actual, buckets=10):
    """Population Stability Index between two numeric distributions."""
    expected = np.asarray(expected, dtype=float)
    actual = np.asarray(actual, dtype=float)
    expected = expected[~np.isnan(expected)]
    actual = actual[~np.isnan(actual)]
    if len(expected) == 0 or len(actual) == 0:
        return 0.0
    combined = np.concatenate([expected, actual])
    if np.min(combined) == np.max(combined):
        return 0.0
    edges = np.unique(np.linspace(np.min(combined), np.max(combined), buckets + 1))
    if len(edges) < 2:
        return 0.0
    e_pct = np.histogram(expected, bins=edges)[0] / len(expected)
    a_pct = np.histogram(actual, bins=edges)[0] / len(actual)
    e_pct = np.where(e_pct == 0, 1e-6, e_pct)
    a_pct = np.where(a_pct == 0, 1e-6, a_pct)
    return float(np.sum((e_pct - a_pct) * np.log(e_pct / a_pct)))


In [ ]:
categorical_cols = [
    "Major Media Code", "Internet Details", "Reception Type Category",
    "JIS_grouped", "Industry Type", "Company Size Category",
    "Gender", "Single/Married Status", "Residence Type", "Name Type",
    "Family Composition Type", "Living Arrangement Type",
    "Insurance Job Type", "Employment Type", "Employment Status Type",
]
categorical_cols = [c for c in categorical_cols if c in train_fe.columns and c in test_fe.columns]
print("Target-encoding columns:", categorical_cols)

te_maps = {}
for col in categorical_cols:
    test_fe[col] = test_fe[col].astype(object)
    oof, mapping, prior = fold_target_encode(train_fe, col, TARGET, n_splits=5, seed=current_seed, smoothing=30)
    train_fe[col + "_te"] = oof
    test_fe[col + "_te"] = test_fe[col].map(lambda x: mapping.get(x, prior))
    te_maps[col] = {"map": mapping, "prior": prior}


In [ ]:
# inspect PSI for every target-encoded column and drop the unstable ones
PSI_DROP_THRESHOLD = 0.25

te_cols = [c + "_te" for c in categorical_cols]
psi_te = {c: psi(train_fe[c].values, test_fe[c].values) for c in te_cols}
psi_te_df = pd.DataFrame(sorted(psi_te.items(), key=lambda x: x[1], reverse=True),
                          columns=["feature", "psi"])
print("PSI by target-encoded feature (train vs test):")
print(psi_te_df.to_string(index=False))

unstable_te = psi_te_df.loc[psi_te_df["psi"] > PSI_DROP_THRESHOLD, "feature"].tolist()
if unstable_te:
    print("\nDropping unstable TE columns (PSI > %.2f):" % PSI_DROP_THRESHOLD, unstable_te)
    train_fe = train_fe.drop(columns=unstable_te)
    test_fe = test_fe.drop(columns=unstable_te)
    te_cols = [c for c in te_cols if c not in unstable_te]
else:
    print("\nAll target-encoded columns are within the PSI threshold.")


## 6. Train / validation split

A time-based 80/20 split (falling back to a stratified random split if dates
aren't usable), so validation AUC reflects performance on applications that
arrive *after* the training window. The raw high-cardinality categorical columns
are excluded in favour of their target-encoded counterparts.


In [ ]:
exclude_cols = {TARGET, ID_COL, "Application Date", "Application Time", "Date of Birth",
                 "application_datetime", "JIS Address Code"}
exclude_cols |= set(categorical_cols)  # raw categoricals are superseded by their _te version

feature_cols = [
    c for c in train_fe.columns
    if c not in exclude_cols
    and c in test_fe.columns
    and str(train_fe[c].dtype) not in ("object", "category")
]
print(f"Final feature set: {len(feature_cols)} columns")

if "Application Date" in train_fe.columns and train_fe["Application Date"].notna().any():
    cutoff = train_fe["Application Date"].quantile(0.80)
    tr_idx = train_fe[train_fe["Application Date"] <= cutoff].index
    val_idx = train_fe[train_fe["Application Date"] > cutoff].index
    if len(val_idx) < 200:
        tr_idx, val_idx = train_test_split(
            train_fe.index, test_size=0.2, random_state=current_seed, stratify=train_fe[TARGET]
        )
else:
    tr_idx, val_idx = train_test_split(
        train_fe.index, test_size=0.2, random_state=current_seed, stratify=train_fe[TARGET]
    )

print("Train rows:", len(tr_idx), " Validation rows:", len(val_idx))

X_tr, y_tr = train_fe.loc[tr_idx, feature_cols], train_fe.loc[tr_idx, TARGET].astype(int)
X_val, y_val = train_fe.loc[val_idx, feature_cols], train_fe.loc[val_idx, TARGET].astype(int)
X_test = test_fe[feature_cols]

median_vals = X_tr.median(numeric_only=True)
X_tr = X_tr.fillna(median_vals)
X_val = X_val.fillna(median_vals)
X_test = X_test.fillna(median_vals)


## 7. PSI check

A train-vs-test stability check on the *exact* feature set the model is about to
use, with the same `psi()` function defined during target encoding. Unstable
target-encoded columns were already removed in step 5; this is the final sanity
check, run before spending time on training, so a drifting feature set is caught
early rather than discovered after the fact.


In [ ]:
psi_rows = []
for c in feature_cols:
    fill_val = median_vals.get(c, 0)
    score = psi(
        train_fe[c].fillna(fill_val).values.astype(float),
        test_fe[c].fillna(fill_val).values.astype(float),
    )
    psi_rows.append((c, score))

psi_final_df = pd.DataFrame(psi_rows, columns=["feature", "psi"]).sort_values("psi", ascending=False)
print("Top 15 features by PSI (train vs test):")
psi_final_df.head(15)


## 8. LightGBM model

In [ ]:
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "seed": current_seed,
    "verbosity": -1,
}

dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

model = lgb.train(
    lgb_params, dtrain, valid_sets=[dtrain, dval], num_boost_round=2000,
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)],
)

val_preds = model.predict(X_val, num_iteration=model.best_iteration)
val_auc = roc_auc_score(y_val, val_preds)
print(f"Validation AUC: {val_auc:.4f}")


## 9. Feature importance

In [ ]:
imp_df = pd.DataFrame({
    "feature": model.feature_name(),
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False).reset_index(drop=True)

top_imp = imp_df.head(15).iloc[::-1]  # reversed so the top feature plots at the top
plt.figure(figsize=(7, 5))
plt.barh(top_imp["feature"], top_imp["gain"])
plt.xlabel("Gain")
plt.title("Top 15 features by LightGBM gain")
plt.tight_layout()
plt.show()

imp_df.head(15)


## 10. SHAP analysis

In [ ]:
# SHAP summary + dependence plots, computed on a validation sample
explainer = shap.TreeExplainer(model)
shap_sample = X_val.sample(n=min(2000, len(X_val)), random_state=current_seed)
shap_values = explainer.shap_values(shap_sample, check_additivity=False)
shap_arr = shap_values[1] if isinstance(shap_values, list) and len(shap_values) == 2 else shap_values

shap.summary_plot(shap_arr, shap_sample, show=True)


In [ ]:
mean_abs_shap = pd.DataFrame({
    "feature": shap_sample.columns,
    "mean_abs_shap": np.abs(shap_arr).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

top_features = mean_abs_shap["feature"].head(2).tolist()
for feat in top_features:
    plt.figure(figsize=(6, 4))
    shap.dependence_plot(feat, shap_arr, shap_sample, show=True)

mean_abs_shap.head(10)


## 11. Calibration curve

In [ ]:
prob_true, prob_pred = calibration_curve(y_val, val_preds, n_bins=10)

plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default rate")
plt.title("Calibration curve (validation)")
plt.legend()
plt.tight_layout()
plt.show()

brier = brier_score_loss(y_val, val_preds)
print(f"Brier score: {brier:.4f}")


## 12. Decile lift

In [ ]:
val_df = pd.DataFrame({
    ID_COL: train_fe.loc[val_idx, ID_COL].values,
    TARGET: y_val.values,
    "pred_prob": val_preds,
})
val_df["decile"] = pd.qcut(val_df["pred_prob"], 10, labels=False, duplicates="drop")

decile_agg = (
    val_df.groupby("decile")
    .agg(n=("pred_prob", "count"), positives=(TARGET, "sum"), avg_pred=("pred_prob", "mean"))
    .reset_index()
)
decile_agg["default_rate"] = decile_agg["positives"] / decile_agg["n"]
overall_rate = val_df[TARGET].mean()
decile_agg["lift"] = decile_agg["default_rate"] / overall_rate
decile_agg = decile_agg.sort_values("decile", ascending=False).reset_index(drop=True)

top_decile_lift = decile_agg.iloc[0]["lift"]
print(f"Top-decile default rate is {top_decile_lift:.2f}x the portfolio average.")
decile_agg


## 13. Business insights

The bullets below are generated directly from this run's SHAP values and decile
lift table — not hand-typed — so they always describe what the model actually
learned, even if you re-run the notebook on a refreshed extract.


In [ ]:
top_n = 5
top_feats = mean_abs_shap["feature"].head(top_n).tolist()
feature_index = {f: i for i, f in enumerate(shap_sample.columns)}

lines = []
for feat in top_feats:
    vals = shap_sample[feat].values.astype(float)
    sv = shap_arr[:, feature_index[feat]]
    mask = ~np.isnan(vals)
    corr = np.corrcoef(vals[mask], sv[mask])[0, 1] if mask.sum() > 1 else np.nan
    if np.isnan(corr):
        direction = "an unclear (mixed) relationship with"
    else:
        direction = "higher predicted default risk as it increases" if corr > 0 else "lower predicted default risk as it increases"
    importance = mean_abs_shap.set_index("feature").loc[feat, "mean_abs_shap"]
    lines.append(
        f"- **{feat}** (mean |SHAP| = {importance:.4f}): associated with {direction} "
        f"(correlation with SHAP value = {corr:.2f})."
    )

lines.append(
    f"\n- The **top predicted-risk decile** defaults at **{top_decile_lift:.2f}x** the portfolio "
    "average rate, so the model's output probability can be used directly to triage applications "
    "into risk tiers rather than only to rank them."
)

display(Markdown("\n".join(lines)))


## 14. Final submission

One final `submission.csv`, predicted from the model trained above (using its
early-stopped `best_iteration`).


In [ ]:
test_preds = model.predict(X_test, num_iteration=model.best_iteration)

submission = pd.DataFrame({
    ID_COL: test_fe[ID_COL],
    TARGET: test_preds,
})

submission_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()


## Executive summary

In [ ]:
summary = f"""\
- **Model:** LightGBM, {len(feature_cols)} features (engineered ratios/flags + PSI-checked target encodings)
- **Validation AUC:** {val_auc:.4f} (time-based hold-out, {len(val_idx)} rows)
- **Brier score:** {brier:.4f}
- **Top SHAP driver:** {mean_abs_shap.iloc[0]['feature']} (mean |SHAP| = {mean_abs_shap.iloc[0]['mean_abs_shap']:.4f})
- **Top-decile lift:** {top_decile_lift:.2f}x the portfolio average default rate
"""
display(Markdown(summary))


*(Copy the bullets above into the Problem Statement section at the top, or into a
project README, once you've run the notebook end to end — that way the headline
numbers shown there are always the ones this exact run actually produced.)*

The CatBoost + XGBoost stacking ensemble explored during experimentation (multiple
seeds, isotonic calibration, averaged submissions) lives separately in
`Credit_Risk_Ensemble_Experiment.ipynb`.
